## Part 1: Preprocessing

In [1]:
# Import our dependencies
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
#  Import and read the attrition data
attrition_df = pd.read_csv('https://static.bc-edx.com/ai/ail-v-1-0/m19/lms/datasets/attrition.csv')
attrition_df.head()

,Age,Attrition,BusinessTravel,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,HourlyRate,JobInvolvement,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,Sales,1,2,Life Sciences,2,94,3,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,Research & Development,8,1,Life Sciences,3,61,2,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,Research & Development,2,2,Other,4,92,2,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,Research & Development,3,4,Life Sciences,4,56,3,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,Research & Development,2,1,Medical,1,40,3,...,3,4,1,6,3,3,2,2,2,2


In [2]:
# Determine the number of unique values in each column
attrition_df.nunique()

,0
Age,43
Attrition,2
BusinessTravel,3
Department,3
DistanceFromHome,29
Education,5
EducationField,6
EnvironmentSatisfaction,4
HourlyRate,71
JobInvolvement,4


In [3]:
# Create y_df with the Attrition and Department columns
y_df = attrition_df[['Attrition', 'Department']]

In [4]:
attrition_df.dtypes

,0
Age,int64
Attrition,object
BusinessTravel,object
Department,object
DistanceFromHome,int64
Education,int64
EducationField,object
EnvironmentSatisfaction,int64
HourlyRate,int64
JobInvolvement,int64


In [5]:
# Create a list of at least 10 column names to use as X data
selected_columns = ['Age', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction','NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']
# Create X_df using your selected columns
X_df = attrition_df[selected_columns]
# Show the data types for X_df
X_df.dtypes

,0
Age,int64
DistanceFromHome,int64
Education,int64
EnvironmentSatisfaction,int64
HourlyRate,int64
JobInvolvement,int64
JobLevel,int64
JobSatisfaction,int64
NumCompaniesWorked,int64
PercentSalaryHike,int64


In [6]:
selected_columns

['Age',
 'DistanceFromHome',
 'Education',
 'EnvironmentSatisfaction',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobSatisfaction',
 'NumCompaniesWorked',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [7]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_df, y_df, test_size=0.2, random_state=42)

In [8]:
# Create a StandardScaler
scaler = StandardScaler()
# Fit the StandardScaler to the training data
X_train_scaled = scaler.fit(X_train)
# Scale the training and testing data
X_test_scaled = scaler.fit(X_test)

X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
len(X_train_scaled)

1176

In [10]:
# Create a OneHotEncoder for the Department column
dept_encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop= None)
# Fit the encoder to the training data
dept_encoder.fit(np.array(y_train[['Department']]).reshape(-1, 1))
# Create two new variables by applying the encoder
# to the training and testing data
dept_train_encoded =  dept_encoder.transform(np.array(y_train[['Department']]).reshape(-1, 1))
dept_test_encoded =  dept_encoder.transform(np.array(y_test[['Department']]).reshape(-1,1))


In [11]:
dept_train_encoded

array([[0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.],
       ...,
       [0., 1., 0.],
       [0., 1., 0.],
       [0., 0., 1.]])

In [12]:
# Create a OneHotEncoder for the Attrition column
attrition_encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore',drop= None)
# Fit the encoder to the training data
attrition_encoder.fit(np.array(y_train[['Attrition']]).reshape(-1, 1))
# Create two new variables by applying the encoder
# to the training and testing data
attrition_train_encoded =  attrition_encoder.transform(np.array(y_train[['Attrition']]).reshape(-1, 1))
attrition_test_encoded =  attrition_encoder.transform(np.array(y_test[['Attrition']]).reshape(-1, 1))

In [13]:
attrition_train_encoded

array([[1., 0.],
       [1., 0.],
       [1., 0.],
       ...,
       [0., 1.],
       [1., 0.],
       [1., 0.]])

## Part 2: Create, Compile, and Train the Model

In [14]:
# Find the number of columns in the X training data.
input_dim = len(X_train.columns)
# Create the input layer
inputs = layers.Input(shape=(input_dim,), name = 'input')
# Create at least two shared layers
shared_layer1 = layers.Dense(64, activation='relu', name = 'shared_layer1')(inputs)
shared_layer2 = layers.Dense(128, activation='relu', name = 'shared_layer2')(shared_layer1)


In [15]:
# Create a branch for Department
# with a hidden layer and an output layer
dept_hidden = layers.Dense(32, activation='relu', name = 'department_hidden')(shared_layer2)
# Create the output layer
dept_output = layers.Dense(dept_train_encoded.shape[1], activation='sigmoid', name='department_output')(dept_hidden)


In [16]:
# Create a branch for Attrition
# with a hidden layer and an output layer
attr_hidden = layers.Dense(32, activation='relu', name = 'attrition_hidden')(shared_layer2)

# Create the output layer
attr_output = layers.Dense(attrition_train_encoded.shape[1], activation='sigmoid', name='attrition_output')(attr_hidden)


In [17]:
# Create the model
model = Model(inputs=inputs, outputs={'department_output':dept_output, 'attrition_output': attr_output})
# Compile the model
model.compile(optimizer='adam',
              loss={'department_output': 'categorical_crossentropy',
                    'attrition_output': 'binary_crossentropy'},
              metrics={'department_output': 'accuracy',
                       'attrition_output': 'accuracy'})
# Summarize the model
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)        │ (None, 20)             │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ shared_layer1 (Dense)     │ (None, 64)             │          1,344 │ input[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ shared_layer2 (Dense)     │ (None, 128)            │          8,320 │ shared_layer1[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attrition_hidden (Dense)  │ (None, 32)             │          4,128 │ shared_layer2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ department_hidden (Dense) │ (None, 32)             │          4,128 │ shared_layer2[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attrition_output (Dense)  │ (None, 2)              │             66 │ attrition_hidden[0][0] │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ department_output (Dense) │ (None, 3)              │             99 │ department_hidden[0][… │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 18,085 (70.64 KB)

 Trainable params: 18,085 (70.64 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
# Train the model
model.fit(X_train_scaled, {'department_output': dept_train_encoded, 'attrition_output': attrition_train_encoded},
          epochs=10, batch_size=32, validation_split=0.2)


Epoch 1/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 6s 18ms/step - attrition_output_accuracy: 0.5996 - attrition_output_loss: 0.6577 - department_output_accuracy: 0.6218 - department_output_loss: 0.8946 - loss: 1.5522 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.4943 - val_department_output_accuracy: 0.6398 - val_department_output_loss: 0.8040 - val_loss: 1.3336
Epoch 2/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - attrition_output_accuracy: 0.8394 - attrition_output_loss: 0.4420 - department_output_accuracy: 0.6786 - department_output_loss: 0.7589 - loss: 1.2014 - val_attrition_output_accuracy: 0.7966 - val_attrition_output_loss: 0.4789 - val_department_output_accuracy: 0.5890 - val_department_output_loss: 0.8030 - val_loss: 1.3141
Epoch 3/10
30/30 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - attrition_output_accuracy: 0.8571 - attrition_output_loss: 0.3915 - department_output_accuracy: 0.6528 - department_output_loss: 0.7360 - loss: 1.1277 - val_attrition_output_accuracy: 0.7966 - va

0.8775510191917419

0.6734693646430969

In [20]:
# Evaluate the model with the testing data
results = model.evaluate(X_test_scaled, {'department_output': dept_test_encoded, 'attrition_output': attrition_test_encoded})
results

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - attrition_output_accuracy: 0.8623 - attrition_output_loss: 0.3574 - department_output_accuracy: 0.6511 - department_output_loss: 0.7911 - loss: 1.1549 


[1.1032097339630127,
 0.7452319860458374,
 0.32335323095321655,
 0.8775510191917419,
 0.6734693646430969]

In [26]:
department_accuracy = results[3]
attrition_accuracy = results[4]
# Print the accuracy for both department and attrition
print(f"Department Accuracy: {department_accuracy}")
print(f"Attrition Accuracy: {attrition_accuracy}")


Department Accuracy: 0.8775510191917419
Attrition Accuracy: 0.6734693646430969


# Summary

In the provided space below, briefly answer the following questions.

1. Is accuracy the best metric to use on this data? Why or why not?

2. What activation functions did you choose for your output layers, and why?

3. Can you name a few ways that this model might be improved?

YOUR ANSWERS HERE

1. no, classes are imbalanced (F1 score is better)
2. we used sigmoid (attrition is binary), softmax is a better fit for department because it is a multi-class problem
3. add more trainning data, add more features, layers, and experimenting

In [28]:
attrition_df[['Attrition']].value_counts()

,count
Attrition,
No,1233
Yes,237


In [29]:
attrition_df[['Department']].value_counts()

,count
Department,
Research & Development,961
Sales,446
Human Resources,63
